# Basic MPCC path following with a kinematic bicycle model.

The planner tracks an S-curve reference trajectory using contouring, lag,
and progress costs with a Savitzky-Golay filter for control smoothing.

In [ ]:
!pip install -q faran faran-visualizer

## Constants

In [ ]:
from faran.numpy import trajectory

REFERENCE = trajectory.waypoints(
    points=[
        (0.0, 0.0),
        (10.0, 0.0),
        (20.0, 10.0),
        (10.0, 20.0),
        (0.0, 20.0),
        (-10.0, 20.0),
        (-20.0, 30.0),
        (-10.0, 40.0),
        (0.0, 40.0),
    ],
    path_length=50.0,
)


HORIZON = 30
TIME_STEP = 0.1
WHEELBASE = 2.5
VEHICLE_WIDTH = 1.2
TEMPERATURE = 50.0
ROLLOUT_COUNT = 256
STEP_LIMIT = 150

## Extractors

In [ ]:
from faran.numpy import types


def heading(states: types.bicycle.StateBatch) -> types.Headings:
    return types.headings(heading=states.heading())

## Setup

In [ ]:
from faran import collectors, metrics
from faran.numpy import extract, filters, model, mppi, sampler, types
from tqdm.auto import tqdm


def create():
    planner, augmented_model, contouring_cost, lag_cost = mppi.mpcc(
        model=model.bicycle.dynamical(
            time_step_size=TIME_STEP,
            wheelbase=WHEELBASE,
            speed_limits=(0.0, 15.0),
            steering_limits=(-0.5, 0.5),
            acceleration_limits=(-3.0, 3.0),
        ),
        sampler=sampler.gaussian(
            standard_deviation=[0.5, 0.2],
            rollout_count=ROLLOUT_COUNT,
            to_batch=types.bicycle.control_input_batch.create,
            seed=42,
        ),
        reference=REFERENCE,
        position_extractor=extract.from_physical(lambda states: states.positions),
        config={
            "weights": {"contouring": 50.0, "lag": 100.0, "progress": 1000.0},
            "virtual": {"velocity_limits": (0.0, 15.0)},
        },
        filter_function=filters.savgol(window_length=11, polynomial_order=3),
    )

    planner = (
        trajectories_collector := collectors.trajectories.decorating(
            state_collector := collectors.states.decorating(
                planner,
                transformer=types.augmented.state_sequence.of_states(
                    physical=types.bicycle.state_sequence.of_states,
                    virtual=types.simple.state_sequence.of_states,
                ),
            ),
            model=augmented_model,
        )
    )

    registry = metrics.registry(
        error_metric := metrics.mpcc_error(contouring=contouring_cost, lag=lag_cost),
        collectors=collectors.registry(state_collector, trajectories_collector),
    )

    return planner, augmented_model, registry, error_metric

## Result

In [ ]:
from dataclasses import dataclass

from faran import MpccErrorMetricResult, access
from faran_visualizer import MpccSimulationResult

type AugmentedState = types.augmented.State[
    types.bicycle.State,
    types.simple.State,
]


@dataclass(frozen=True)
class Result:
    """Outcome of a planning simulation."""

    final_state: AugmentedState
    visualization: MpccSimulationResult
    tracking_errors: MpccErrorMetricResult


    @property
    def progress(self) -> float:
        return float(self.final_state.virtual.array[0])

    @property
    def reached_goal(self) -> bool:
        return self.progress >= REFERENCE.path_length * 0.9

    @property
    def collision_detected(self) -> bool:
        # This example doesn't have any obstacles, so we'll just return False here.
        return False


def extract_simulation_results(current_state, registry, error_metric):
    trajectories = registry.data(access.trajectories.require())
    errors = registry.get(error_metric)

    visualization = MpccSimulationResult(
        reference=REFERENCE,
        states=registry.data(access.states.require()),
        optimal_trajectories=[it.optimal for it in trajectories],
        nominal_trajectories=[it.nominal for it in trajectories],
        contouring_errors=errors.contouring,
        lag_errors=errors.lag,
        time_step_size=TIME_STEP,
        wheelbase=WHEELBASE,
        vehicle_width=VEHICLE_WIDTH,
        max_contouring_error=2.5,
        max_lag_error=5.0,
    )

    return Result(
        final_state=current_state, visualization=visualization, tracking_errors=errors
    )

## Simulation loop

In [ ]:
def run(planner, augmented_model, registry, error_metric):
    current_state = types.augmented.state.of(
        physical=types.bicycle.state.create(x=0.0, y=0.0, heading=0.0, speed=0.0),
        virtual=types.simple.state.zeroes(dimension=1),
    )
    nominal = types.augmented.control_input_sequence.of(
        physical=types.bicycle.control_input_sequence.zeroes(horizon=HORIZON),
        virtual=types.simple.control_input_sequence.zeroes(
            horizon=HORIZON, dimension=1
        ),
    )

    bar = tqdm(range(STEP_LIMIT), desc="Simulation", unit="step")
    for step in bar:
        control = planner.step(
            temperature=TEMPERATURE,
            nominal_input=nominal,
            initial_state=current_state,
        )
        nominal = control.nominal
        current_state = augmented_model.step(
            inputs=control.optimal, state=current_state
        )

        if current_state.virtual.array[0] >= REFERENCE.path_length * 0.9:
            bar.write(f"Reached goal at step {step + 1}.")
            break

        bar.set_postfix(progress=f"{current_state.virtual.array[0]:.2f}%")

    return extract_simulation_results(current_state, registry, error_metric)

## Visualization

In [ ]:
from faran_visualizer import configure, visualizer


async def visualize(result: Result) -> None:
    configure(output_directory=".")
    await visualizer.mpcc()(result.visualization, key="visualization")

## Run

In [ ]:
import asyncio

planner, augmented_model, registry, error_metric = create()
result = run(planner, augmented_model, registry, error_metric)
print(f"Path progress: {result.progress:.1f} / {REFERENCE.path_length}")
print(f"Reached goal: {result.reached_goal}")
print(f"Collision detected: {result.collision_detected}")
await visualize(result)

In [ ]:
from IPython.display import IFrame, display as show_inline
show_inline(IFrame("mpcc-simulation/visualization.html", width="100%", height=600))